In [2]:
# ============================================================
# #  GroupDNA — WhatsApp Group Chat Decoder
#
# **Project:** GroupDNA — Minor Project (Week 1)
# **Course:** The Unlox Academy
# **Dataset:** hostel_bois.txt (Synthetic WhatsApp Export)
#
# ---
# ### What this notebook does
# Reads a WhatsApp group chat export file and produces a full analytics report:
# - Parses every message, handling edge cases (system messages, media, deleted messages)
# - Group overview with per-person message counts
# - Most active day and busiest hour detection
# - Activity heatmap using NumPy (person × hour matrix)
# - Top 10 most-used words with bar chart
# - Response speed and longest silent streaks
# - Personality archetype detection for every member
# - Final formatted report ready to screenshot
#
# **Constraint:** No pandas, no matplotlib, no regex, no Counter. Pure Python + NumPy only.

# ============================================================
# ## Setup — Imports
# Only two imports are allowed: `numpy` and `datetime`. Everything else is built from scratch.

import numpy as np
from datetime import datetime

# File path — change this if your file is in a different location
# In Google Colab: upload hostel_bois.txt using the folder icon, then use '/content/hostel_bois.txt'
FILE_PATH = 'hostel_bois.txt'

print('Imports ready. NumPy version:', np.__version__)


# ============================================================
# ---
# ## Feature 1 — The Chat Parser
#
# Reads the file line by line and extracts three things from each valid message line:
# 1. **Timestamp** — the date and time string (e.g. `12/04/24, 23:14`)
# 2. **Sender** — the person's name
# 3. **Message text** — what they actually said
#
# ### Edge cases handled:
# - **System messages** — lines with no `sender: message` format (group creation, encryption notice, etc.)
# - **Media omitted** — counted per person but excluded from text analysis
# - **Deleted messages** — counted per person separately
# - **Multi-line messages** — lines not starting with date pattern are treated as continuations
# - **Empty lines** — silently skipped
#
# **Key string techniques used:** `str.split(' - ', 1)` to separate timestamp from rest, then `str.split(': ', 1)` to separate sender from message. The `1` argument ensures only the FIRST occurrence is split — critical so messages containing ` - ` or `: ` don't break the parser.

def parse_chat(filename):
    """
    Reads a WhatsApp .txt export and returns:
    - messages: list of dicts, each with 'timestamp', 'sender', 'text' keys
    - system_count: number of system message lines skipped
    - media_count: total <Media omitted> entries
    - deleted_count: total 'This message was deleted' entries
    - media_per_person: dict {sender: media_count}
    - deleted_per_person: dict {sender: deleted_count}
    """
    messages = []
    system_count = 0
    media_count = 0
    deleted_count = 0
    media_per_person = {}
    deleted_per_person = {}

    # Step 1: Read the entire file
    with open(filename, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    # Step 2: Loop over every line and parse
    for line in lines:
        line = line.strip()  # Remove leading/trailing whitespace and newlines

        # Skip empty lines
        if not line:
            continue

        # Check if this line starts with a date pattern: DD/MM/YY
        # A valid timestamp line has '/' at position 2 and 5
        if not (len(line) >= 8 and line[2] == '/' and line[5] == '/'):
            # This is a continuation line of a multi-line message — skip for now
            # (In a real chat you'd append it to the previous message)
            continue

        # Check that ' - ' separator exists (it always does in valid WhatsApp lines)
        if ' - ' not in line:
            continue

        # Split: 'DD/MM/YY, HH:MM - <rest>'  =>  [timestamp, rest]
        # The '1' means split only at the FIRST occurrence
        parts = line.split(' - ', 1)
        timestamp = parts[0].strip()  # e.g. '12/04/24, 23:14'
        rest = parts[1]               # e.g. 'Rahul: bhai kya scene'

        # If there's no ': ' in rest, it's a system message (no sender colon)
        if ': ' not in rest:
            system_count += 1
            continue

        # Split: 'Rahul: bhai kya scene'  =>  ['Rahul', 'bhai kya scene']
        sender_msg = rest.split(': ', 1)
        sender = sender_msg[0].strip()
        msg = sender_msg[1].strip()

        # Handle media omitted
        if msg == '<Media omitted>':
            media_count += 1
            media_per_person[sender] = media_per_person.get(sender, 0) + 1
            continue

        # Handle deleted messages
        if msg == 'This message was deleted':
            deleted_count += 1
            deleted_per_person[sender] = deleted_per_person.get(sender, 0) + 1
            continue

        # Valid real message — store as a dictionary
        messages.append({
            'timestamp': timestamp,
            'sender': sender,
            'text': msg
        })

    return messages, system_count, media_count, deleted_count, media_per_person, deleted_per_person


# Run the parser
messages, sys_cnt, med_cnt, del_cnt, media_pp, deleted_pp = parse_chat(FILE_PATH)

# Convert timestamps to datetime objects (needed later for time analysis)
# datetime.strptime parses a string using a format code:
# %d = day, %m = month, %y = 2-digit year, %H = hour (24h), %M = minute
for m in messages:
    m['dt'] = datetime.strptime(m['timestamp'], '%d/%m/%y, %H:%M')

# Derive key info
participants = sorted(set(m['sender'] for m in messages))  # Sorted alphabetically
person_index = {p: i for i, p in enumerate(participants)}   # {'Aman': 0, 'Karan': 1, ...}
total_msgs = len(messages)
first_dt = min(m['dt'] for m in messages)
last_dt = max(m['dt'] for m in messages)
days_total = (last_dt - first_dt).days + 1

# Helper: get all messages for a given person
def msgs_of(person):
    return [m for m in messages if m['sender'] == person]

# Checkpoint print
print(f'Successfully parsed {total_msgs} messages from {len(participants)} participants over {days_total} days')
print(f'Skipped: {sys_cnt} system messages, {med_cnt} media-omitted, {del_cnt} deleted messages')
print(f'Participants: {", ".join(participants)}')
print(f'\nFirst 3 messages:')
for m in messages[:3]:
    print(f'  [{m["timestamp"]}] {m["sender"]}: {m["text"][:60]}')
print(f'\nLast 3 messages:')
for m in messages[-3:]:
    print(f'  [{m["timestamp"]}] {m["sender"]}: {m["text"][:60]}')


# ============================================================
# ---
# ## Feature 2 — Group Overview
#
# Computes headline statistics: total messages, date range, participants, and a per-person message count sorted highest to lowest.
#
# **Concepts used:** Dictionaries for per-person counts, `sorted()` with `key=` for ranking, f-strings with width specifiers for alignment.

def compute_overview(messages, participants, days_total, first_dt, last_dt):
    """Compute and return per-person message counts and group stats."""
    # Count messages per person using a dictionary
    msg_counts = {}
    for m in messages:
        msg_counts[m['sender']] = msg_counts.get(m['sender'], 0) + 1

    # Sort by count descending
    sorted_counts = sorted(msg_counts.items(), key=lambda x: x[1], reverse=True)

    return msg_counts, sorted_counts


def print_overview(messages, participants, days_total, first_dt, last_dt, total_msgs):
    """Print the Group Overview section."""
    msg_counts, sorted_counts = compute_overview(messages, participants, days_total, first_dt, last_dt)

    # Build bar chart per person
    max_count = sorted_counts[0][1]
    bar_max_len = 20

    print('=' * 60)
    print(f'{" GROUP OVERVIEW":^60}')
    print('=' * 60)
    print(f'  {"Group":<20}: Hostel Bois 4ever')
    print(f'  {"Period":<20}: {first_dt.strftime("%d %B %Y")} to {last_dt.strftime("%d %B %Y")} ({days_total} days)')
    print(f'  {"Total messages":<20}: {total_msgs:,}')
    print(f'  {"Participants":<20}: {len(participants)}')
    print()
    print(f'  {"MESSAGES PER PERSON"}')
    print(f'  {"-" * 56}')
    for name, count in sorted_counts:
        pct = count / total_msgs * 100
        bar_len = int(count / max_count * bar_max_len)
        bar = '█' * bar_len + '.' * (bar_max_len - bar_len) if bar_len > 0 else '.' * bar_max_len
        if bar_len == 0:
            bar = '.'
        print(f'  {name:<8} {bar:<22} {count:>4} ({pct:>4.1f}%)')
    print()

    return msg_counts


msg_counts = print_overview(messages, participants, days_total, first_dt, last_dt, total_msgs)


# ============================================================
# ---
# ## Feature 3 — Most Active Day and Hour
#
# Identifies the single busiest day (most messages) and the most active hour of the day.
#
# **Concepts used:** Dictionary with date/hour as key and count as value, `max()` with `key=` parameter.

def compute_active_periods(messages, days_total):
    """Find the busiest day and busiest hour."""

    # Count messages per calendar day
    # Key: 'DD Month YYYY' string, Value: message count
    day_counts = {}
    for m in messages:
        day_key = m['dt'].strftime('%d %B %Y')  # e.g. '14 April 2024'
        day_counts[day_key] = day_counts.get(day_key, 0) + 1

    # Find the day with the maximum count
    busiest_day = max(day_counts, key=lambda d: day_counts[d])
    busiest_day_count = day_counts[busiest_day]

    # Count messages per hour (0 to 23), summed across all days
    hour_counts = {}
    for m in messages:
        h = m['dt'].hour
        hour_counts[h] = hour_counts.get(h, 0) + 1

    # Find the peak hour
    busiest_hour = max(hour_counts, key=lambda h: hour_counts[h])
    avg_per_day = hour_counts[busiest_hour] / days_total  # Average over 60 days

    return busiest_day, busiest_day_count, busiest_hour, avg_per_day, hour_counts


busiest_day, busiest_day_count, busiest_hour, avg_per_day, hour_counts = compute_active_periods(messages, days_total)

print(f'  Busiest day  : {busiest_day} ({busiest_day_count} messages)')
print(f'  Busiest hour : {busiest_hour:02d}:00 - {busiest_hour+1:02d}:00 (avg {avg_per_day:.1f} messages per day)')


# ============================================================
# ---
# ## Feature 4 — Activity Heatmap (NumPy)
#
# Builds a 2D NumPy matrix: **rows = participants**, **columns = hours 0–23**.
# Each cell `[person_index, hour]` stores the total messages that person sent during that hour.
#
# Then renders it as a text-art heatmap using 4 shading levels:
# - `. ` → 0–25% of that person's max
# - `░ ` → 25–50%
# - `▒ ` → 50–75%
# - `█ ` → 75–100%
#
# **Concepts used:** `np.zeros()` to create the matrix, integer indexing `[row, col]` to increment cells, `np.sum()` and `.max()` for aggregation, slicing for night hours.

def build_heatmap(messages, participants, person_index):
    """
    Build a (num_participants × 24) NumPy matrix.
    heatmap[i, h] = number of messages person i sent during hour h.
    """
    # Create a matrix of zeros: shape (6 persons, 24 hours)
    heatmap = np.zeros((len(participants), 24), dtype=int)

    # Fill in the matrix by iterating over every message
    for m in messages:
        row = person_index[m['sender']]  # Which person (row index)
        col = m['dt'].hour               # Which hour (column index)
        heatmap[row, col] += 1           # Increment that cell

    return heatmap


def render_heatmap(heatmap, participants):
    """Print the heatmap as terminal block art."""
    # 4 block levels: 0-25%, 25-50%, 50-75%, 75-100%
    blocks = ['. ', '░ ', '▒ ', '█ ']

    # Header row: show every 3rd hour
    display_hours = [0, 3, 6, 9, 12, 15, 18, 21]
    print(f'  {"ACTIVITY HEATMAP (hour of day — every 3rd hour shown)"}')
    print(f'  {"-" * 56}')
    print(f'  {"":9}', end='')
    for h in display_hours:
        print(f'{h:02d}    ', end='')
    print()

    for i, person in enumerate(participants):
        row = heatmap[i]            # 1D array of 24 values for this person
        row_max = row.max()         # Their peak hour count

        print(f'  {person:<9}', end='')

        # Only render the 8 display hours (every 3rd hour)
        for h in display_hours:
            if row_max == 0:
                block = blocks[0]
            else:
                frac = row[h] / row_max  # Fraction of their personal maximum
                if frac == 0:
                    block = blocks[0]
                elif frac < 0.25:
                    block = blocks[0]
                elif frac < 0.5:
                    block = blocks[1]
                elif frac < 0.75:
                    block = blocks[2]
                else:
                    block = blocks[3]
            print(f'{block}    ', end='')
        print()
    print()


# Build and display the heatmap
heatmap = build_heatmap(messages, participants, person_index)

print('Heatmap matrix (NumPy array, 6 persons × 24 hours):')
print(f'Shape: {heatmap.shape}')
print(f'Total messages in matrix: {heatmap.sum()} (should match parsed total: {total_msgs})')
print()
render_heatmap(heatmap, participants)

# Checkpoint: verify Aman night owl %
aman_idx = person_index['Aman']
# Night hours = 23, 0, 1, 2, 3, 4
night_hours = list(range(23, 24)) + list(range(0, 5))
aman_night_msgs = sum(heatmap[aman_idx, h] for h in night_hours)
aman_total = int(heatmap[aman_idx].sum())
print(f'  Checkpoint — Aman night hours (23:00–04:59): {aman_night_msgs}/{aman_total} = {aman_night_msgs/aman_total*100:.1f}% (expected ~80%)')


# ============================================================
# ---
# ## Feature 5 — Top Words
#
# Tokenizes every message, normalizes to lowercase, strips punctuation, filters stop words, and counts every word using a plain dictionary (no `Counter` allowed).
#
# Renders the top 10 with a proportional block bar chart.
#
# **Concepts used:** `str.lower()`, `str.split()`, `str.strip()` with punctuation chars, dictionary for counting, `sorted()` with `reverse=True`.

# Stop words — common English words that add no meaning to group analysis
# You can expand this list; these are the ones that gave clean results on this dataset
STOP_WORDS = {
    'i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for',
    'it', 'me', 'my', 'you', 'your', 'we', 'he', 'she', 'they', 'was',
    'are', 'be', 'this', 'that', 'with', 'at', 'but', 'not', 'so',
    'do', 'have', 'has', 'had', 'its', 'an', 'as', 'if', 'by', 'up',
    'no', 'yeah', 'ok', 'okay', 'hi', 'yes', 'will', 'just', 'from',
    'im', 'were', 'can', 'also', 'there', 'about', 'what', 'when', 'which',
    'how', 'am', 'his', 'her', 'all', 'one', 'out', 'now', 'been', 'did',
    'get', 'got', 'our', 'us', 'them', 'their', 'who', 'too', 'would',
    'could', 'should', 'because', 'very', 'more', 'like', 'some', 'than',
    'then', 'these', 'those', 'other', 'into', 'over', 'after', 'before',
    'any', 'even', 'much', 'many', 'why', 'where', 'still', 'already',
    'every', 'each', 'both', 'same', 'back', 'down', 'only', 'new', 'see',
    'know', 'think', 'want', 'need', 'go', 'going', 'come', 'came', 'make',
    'made', 'time', 'said', 'say', 'right', 'well', 'here', 'again',
    'through', 'being', 'between', 're', 'telling', 'told', 'wa',
    'actually', 'really', 'always', 'never', 'something', 'everything',
    'started', 'entire'
}

# Characters to strip from word edges
PUNCT = '.,!?;:\'\'"()[]{}@#$%^&*-_+=<>/\\|`~'


def tokenize(text):
    """Split a message into clean word tokens."""
    tokens = []
    for word in text.lower().split():
        word = word.strip(PUNCT)
        # Keep only alphabetic words (filters emojis, numbers, URLs)
        # Also filter very short words and stop words
        if len(word) > 1 and word not in STOP_WORDS and word.replace("'", '').isalpha():
            tokens.append(word)
    return tokens


def compute_word_freq(messages):
    """Build a word frequency dictionary for the whole group."""
    word_freq = {}  # {word: count} — built manually, no Counter
    for m in messages:
        for word in tokenize(m['text']):
            word_freq[word] = word_freq.get(word, 0) + 1
    return word_freq


def compute_word_freq_per_person(messages, participants):
    """Build per-person word frequency dictionaries."""
    person_word_freq = {p: {} for p in participants}
    for m in messages:
        p = m['sender']
        for word in tokenize(m['text']):
            person_word_freq[p][word] = person_word_freq[p].get(word, 0) + 1
    return person_word_freq


def print_top_words(word_freq, top_n=10):
    """Print top N words with a proportional bar chart."""
    top_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)[:top_n]
    max_count = top_words[0][1] if top_words else 1
    bar_max = 24

    print(f'  {"THIS GROUP\'S FAVOURITE WORDS"}')
    print(f'  {"-" * 56}')
    for word, count in top_words:
        bar_len = int(count / max_count * bar_max)
        bar = '█' * bar_len
        print(f'  {word:<12} {bar:<26} {count}')
    print()


# Compute and display
word_freq = compute_word_freq(messages)
person_word_freq = compute_word_freq_per_person(messages, participants)

print_top_words(word_freq)

# Bonus: top 3 words per person
print(f'  TOP 3 WORDS PER PERSON')
print(f'  {"-" * 56}')
for p in participants:
    top3 = sorted(person_word_freq[p].items(), key=lambda x: x[1], reverse=True)[:3]
    top3_str = ', '.join(f'{w} ({c})' for w, c in top3)
    print(f'  {p:<8} → {top3_str}')
print()


# ============================================================
# ---
# ## Feature 6 — Response Speed & Silent Streaks
#
# Two sub-metrics:
# 1. **Average response time** — for each person, how long on average do they take to reply after someone else speaks?
# 2. **Longest silent streak** — the longest consecutive run of days where a person sent zero messages.
#
# **New concept used:** `datetime.strptime()` to parse timestamp strings. Subtracting two `datetime` objects gives a `timedelta`. Calling `.total_seconds()` on that gives the gap in seconds.

def compute_response_times(messages, participants):
    """
    For each person, compute average time (in seconds) between:
    - last message from anyone ELSE
    - their own next message
    Only gaps under 3 hours are counted (to avoid counting 'woke up the next day' as a response).
    """
    MAX_GAP_SECONDS = 3 * 3600  # 3 hours
    gaps_per_person = {p: [] for p in participants}  # {person: [gap_in_seconds, ...]}

    # Walk through messages, track the last message time before this person speaks
    last_other_time = None  # Timestamp of last message from a DIFFERENT person
    last_sender = None

    for m in messages:
        person = m['sender']
        current_time = m['dt']

        # If the previous message was from someone else, this is a response
        if last_sender is not None and last_sender != person and last_other_time is not None:
            gap = (current_time - last_other_time).total_seconds()
            if 0 < gap < MAX_GAP_SECONDS:  # Only count reasonable response windows
                gaps_per_person[person].append(gap)

        # Update last_other_time whenever someone different speaks
        if last_sender != person:
            last_other_time = current_time
        last_sender = person

    # Compute averages
    avg_response = {}
    for p in participants:
        gaps = gaps_per_person[p]
        avg_response[p] = sum(gaps) / len(gaps) if gaps else float('inf')

    return avg_response


def compute_silent_streaks(messages, participants, first_dt, last_dt):
    """
    For each person, find the longest consecutive run of days with zero messages.
    """
    from datetime import timedelta

    # Build set of active dates per person
    active_dates = {p: set() for p in participants}
    for m in messages:
        active_dates[m['sender']].add(m['dt'].date())

    # Find longest silent streak for each person
    longest_streaks = {}
    streak_dates = {}

    for p in participants:
        max_streak = 0
        current_streak = 0
        streak_start = None
        best_start = None
        best_end = None
        current_start = None

        # Walk every day in the chat period
        current_date = first_dt.date()
        end_date = last_dt.date()

        while current_date <= end_date:
            if current_date not in active_dates[p]:
                # Silent day
                if current_streak == 0:
                    current_start = current_date  # Start of a new streak
                current_streak += 1
                if current_streak > max_streak:
                    max_streak = current_streak
                    best_start = current_start
                    best_end = current_date
            else:
                # Active day — reset streak
                current_streak = 0

            current_date += timedelta(days=1)

        longest_streaks[p] = max_streak
        streak_dates[p] = (best_start, best_end) if max_streak > 0 else (None, None)

    return longest_streaks, streak_dates, active_dates


def format_duration(seconds):
    """Format seconds into a human-readable string."""
    if seconds == float('inf'):
        return 'N/A'
    if seconds < 60:
        return f'{seconds:.0f} sec'
    elif seconds < 3600:
        return f'{seconds/60:.1f} min'
    else:
        return f'{seconds/3600:.1f} hrs'


# Compute
avg_response = compute_response_times(messages, participants)
longest_streaks, streak_dates, active_dates = compute_silent_streaks(messages, participants, first_dt, last_dt)

# Print response patterns
sorted_response = sorted(avg_response.items(), key=lambda x: x[1])
fastest = sorted_response[0]
slowest = sorted_response[-1]

print(f'  RESPONSE PATTERNS')
print(f'  {"-" * 56}')
print(f'  Fastest replier : {fastest[0]} (avg {format_duration(fastest[1])})')
print(f'  Slowest replier : {slowest[0]} (avg {format_duration(slowest[1])})')
print()

# Print silent streaks
sorted_streaks = sorted(longest_streaks.items(), key=lambda x: x[1], reverse=True)
print(f'  LONGEST SILENT STREAKS (consecutive days with zero messages)')
print(f'  {"-" * 56}')
for person, streak in sorted_streaks:
    start, end = streak_dates[person]
    active_count = len(active_dates[person])
    if streak == 0:
        print(f'  {person:<8} : {streak} days (never went silent)')
    elif start and end:
        print(f'  {person:<8} : {streak} days ({start.strftime("%d %b")} to {end.strftime("%d %b")})')
    else:
        print(f'  {person:<8} : {streak} days')
print()


# ============================================================
# ---
# ## Feature 7 — Personality Archetype Detection
#
# Eight archetypes, each with a quantitative detection rule. Every person gets exactly ONE archetype — the one they score highest on (relative to other participants).
#
# ### Assignment algorithm:
# 1. Compute each person's raw score on all 8 archetypes
# 2. Normalize each archetype's scores to 0–1 (using that archetype's max across all participants)
# 3. Each person's "best archetype" is the one with their highest normalized score
# 4. Sort persons by their confidence (how dominant their best score is)
# 5. Assign greedily: highest-confidence person gets their preferred archetype first
#
# **Tie-breaking rule:** If two people tie for the same archetype, the one with the higher raw score wins. The other falls back to their second-best archetype. This is documented as a comment in the code.

# ── ARCHETYPE SCORING FUNCTIONS ──────────────────────────────────────
# Each function takes a person's name and returns a numeric score.
# Higher score = stronger fit for that archetype.

def score_spammer(person):
    """
    THE SPAMMER: average consecutive message burst length.
    A 'burst' is a run of messages from this person without anyone else speaking.
    """
    bursts = []
    current_burst = 0
    for m in messages:
        if m['sender'] == person:
            current_burst += 1
        else:
            if current_burst > 0:
                bursts.append(current_burst)
                current_burst = 0
    if current_burst > 0:  # Don't forget the last burst
        bursts.append(current_burst)
    return sum(bursts) / len(bursts) if bursts else 0


def score_night_owl(person):
    """
    THE NIGHT OWL: % of messages sent between 23:00 and 04:59.
    Uses the heatmap matrix (already built with NumPy).
    """
    idx = person_index[person]
    # Night hours: 23, 0, 1, 2, 3, 4
    night = sum(heatmap[idx, h] for h in list(range(23, 24)) + list(range(0, 5)))
    total = int(heatmap[idx].sum())
    return (night / total * 100) if total > 0 else 0


# Caring keywords used for Group Mom detection
CARING_KEYWORDS = [
    'okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please',
    'reminder', 'drink water', "don't forget", 'careful', 'rest', 'home'
]

def score_group_mom(person):
    """
    THE GROUP MOM: total count of caring keyword occurrences in their messages.
    """
    score = 0
    for m in msgs_of(person):
        text = m['text'].lower()
        for kw in CARING_KEYWORDS:
            score += text.count(kw)
    return score


def score_storyteller(person):
    """
    THE STORYTELLER: average words per message.
    """
    person_msgs = msgs_of(person)
    if not person_msgs:
        return 0
    total_words = sum(len(m['text'].split()) for m in person_msgs)
    return total_words / len(person_msgs)


def score_drama_queen(person):
    """
    THE DRAMA QUEEN: % of messages that are entirely uppercase (min 3 alpha chars)
    OR contain 2+ exclamation marks.
    """
    person_msgs = msgs_of(person)
    if not person_msgs:
        return 0
    drama_count = 0
    for m in person_msgs:
        text = m['text']
        alpha_chars = ''.join(c for c in text if c.isalpha())
        # All-caps check: only count if message has at least 3 alphabetic chars
        is_all_caps = len(alpha_chars) >= 3 and text.upper() == text
        has_double_exclaim = text.count('!') >= 2
        if is_all_caps or has_double_exclaim:
            drama_count += 1
    return drama_count / len(person_msgs) * 100


def score_ghost(person):
    """
    THE GHOST: % of days in the chat period where the person sent zero messages.
    """
    return (days_total - len(active_dates[person])) / days_total * 100


def score_comedian(person):
    """
    THE COMEDIAN: % of messages containing a laughter keyword.
    """
    person_msgs = msgs_of(person)
    if not person_msgs:
        return 0
    laugh_words = {'lol', 'lmao', 'haha', 'rofl', 'lmfao', 'hehe', 'xd'}
    funny = sum(
        1 for m in person_msgs
        if any(w in m['text'].lower().split() for w in laugh_words)
    )
    return funny / len(person_msgs) * 100


def score_question_master(person):
    """
    THE QUESTION MASTER: % of messages ending with '?'.
    """
    person_msgs = msgs_of(person)
    if not person_msgs:
        return 0
    questions = sum(1 for m in person_msgs if m['text'].strip().endswith('?'))
    return questions / len(person_msgs) * 100


# ── BONUS ARCHETYPE ───────────────────────────────────────────────────
# THE HYPE MACHINE: person who uses hype words ('fire', 'lit', 'goat', 'legend',
# 'bro', 'king', 'badass', 'epic') most frequently as a % of their messages.
# Detection: % of messages containing at least one hype word.

def score_hype_machine(person):
    """
    THE HYPE MACHINE: % of messages with hype/energy words.
    Typical of someone who cheers everyone up with high-energy language.
    """
    person_msgs = msgs_of(person)
    if not person_msgs:
        return 0
    hype_words = {'fire', 'lit', 'goat', 'legend', 'bro', 'king', 'badass',
                  'epic', 'sahi', 'mast', 'chal', 'bhai', 'dude'}
    hyped = sum(
        1 for m in person_msgs
        if any(w in m['text'].lower().split() for w in hype_words)
    )
    return hyped / len(person_msgs) * 100


# ── ARCHETYPE REGISTRY ────────────────────────────────────────────────
ARCHETYPES = {
    'THE SPAMMER':        score_spammer,
    'THE GROUP MOM':      score_group_mom,
    'THE NIGHT OWL':      score_night_owl,
    'THE STORYTELLER':    score_storyteller,
    'THE DRAMA QUEEN':    score_drama_queen,
    'THE GHOST':          score_ghost,
    'THE COMEDIAN':       score_comedian,
    'THE QUESTION MASTER':score_question_master,
    'THE HYPE MACHINE':   score_hype_machine,  # Bonus archetype
}


def assign_archetypes(participants, archetypes):
    """
    Assign one archetype to each person using a greedy exclusive algorithm.

    Algorithm:
    1. Compute raw scores for all (person, archetype) pairs.
    2. Normalize scores within each archetype (divide by that archetype's maximum).
    3. Each person's 'confidence' = their best normalized score.
    4. Sort persons by confidence descending. Most certain fits go first.
    5. Assign each person their highest-scoring unassigned archetype.

    Tie-breaking rule: If two people have equal normalized scores for the same
    archetype, the one with the higher RAW score wins. The other falls back
    to their second-best archetype.
    """
    # Step 1: Raw scores
    raw_scores = {
        p: {arch: func(p) for arch, func in archetypes.items()}
        for p in participants
    }

    # Step 2: Normalize per archetype
    arch_maxes = {
        arch: max(raw_scores[p][arch] for p in participants)
        for arch in archetypes
    }
    norm_scores = {
        p: {
            arch: (raw_scores[p][arch] / arch_maxes[arch]) if arch_maxes[arch] > 0 else 0
            for arch in archetypes
        }
        for p in participants
    }

    # Step 3: Each person's best archetype and confidence
    person_best = {
        p: max(norm_scores[p], key=norm_scores[p].get)
        for p in participants
    }
    person_conf = {
        p: norm_scores[p][person_best[p]]
        for p in participants
    }

    # Step 4 & 5: Greedy assignment — most confident person picks first
    used_archetypes = set()
    assigned = {}  # {person: (archetype_name, raw_score)}

    for person in sorted(participants, key=lambda p: -person_conf[p]):
        # Try archetypes in order of this person's normalized score (highest first)
        for arch in sorted(norm_scores[person], key=norm_scores[person].get, reverse=True):
            if arch not in used_archetypes:
                assigned[person] = (arch, raw_scores[person][arch])
                used_archetypes.add(arch)
                break

    return assigned, raw_scores


# Run archetype detection
assigned_archetypes, all_raw_scores = assign_archetypes(participants, ARCHETYPES)

# Print results
print(f'  PERSONALITY ARCHETYPES')
print(f'  {"-" * 56}')

archetype_descriptions = {
    'THE SPAMMER':         lambda p, s: f'avg {s:.1f} msgs in a row',
    'THE GROUP MOM':       lambda p, s: f'caring keyword score: {s:.0f}',
    'THE NIGHT OWL':       lambda p, s: f'{s:.1f}% msgs between 23h-04h',
    'THE STORYTELLER':     lambda p, s: f'avg {s:.1f} words per msg',
    'THE DRAMA QUEEN':     lambda p, s: f'{s:.1f}% ALL-CAPS / double-exclaim msgs',
    'THE GHOST':           lambda p, s: f'silent on {int(s/100*days_total)} of {days_total} days',
    'THE COMEDIAN':        lambda p, s: f'{s:.1f}% msgs with laughter words',
    'THE QUESTION MASTER': lambda p, s: f'{s:.1f}% msgs end with "?"',
    'THE HYPE MACHINE':    lambda p, s: f'{s:.1f}% msgs with hype/energy words',
}

for person in participants:
    arch, raw_score = assigned_archetypes[person]
    desc_func = archetype_descriptions.get(arch, lambda p, s: f'score: {s:.2f}')
    desc = desc_func(person, raw_score)
    print(f'  {person:<8} → {arch} ({desc})')
print()


# ============================================================
# ---
# ## Feature 8 — The Final Report
#
# Assembles everything into one clean, formatted printed report.
# The formatting uses:
# - `'=' * 60` for section dividers
# - `f'{name:<10}'` for left-aligned padding (fixed-width columns)
# - `f'{value:>5}'` for right-aligned numbers
# - Block characters (`█`, `▒`, `░`) for visual bars and heatmap
#
# **Goal:** The output should look like something you'd screenshot and post on LinkedIn.

def print_final_report():
    """Print the complete GroupDNA report in one clean call."""

    W = 62  # Report width

    # ── HEADER ──────────────────────────────────────────────────
    print('=' * W)
    print(f'{" 🧬 GROUPDNA REPORT — Hostel Bois 4ever ":^{W}}')
    print(f'{f" {days_total} days  •  {total_msgs:,} messages  •  {len(participants)} members ":^{W}}')
    print('=' * W)
    print()

    # ── SECTION 1: GROUP OVERVIEW ────────────────────────────────
    print(f'  ── GROUP OVERVIEW {"─" * (W - 20)}')
    print(f'  Period        : {first_dt.strftime("%d %B %Y")} to {last_dt.strftime("%d %B %Y")}')
    print(f'  Busiest day   : {busiest_day} ({busiest_day_count} messages)')
    print(f'  Busiest hour  : {busiest_hour:02d}:00 – {busiest_hour+1:02d}:00')
    print()

    # ── SECTION 2: MESSAGES PER PERSON ──────────────────────────
    print(f'  ── MESSAGES PER PERSON {"─" * (W - 25)}')
    sorted_counts = sorted(msg_counts.items(), key=lambda x: x[1], reverse=True)
    max_count = sorted_counts[0][1]
    for name, count in sorted_counts:
        pct = count / total_msgs * 100
        bar_len = int(count / max_count * 20)
        bar = '█' * bar_len if bar_len > 0 else '.'
        print(f'  {name:<8}  {bar:<22}  {count:>4}  ({pct:>4.1f}%)')
    print()

    # ── SECTION 3: ACTIVITY HEATMAP ─────────────────────────────
    print(f'  ── ACTIVITY HEATMAP (hour of day, every 3 hours) {"─" * (W - 52)}')
    display_hours = [0, 3, 6, 9, 12, 15, 18, 21]
    blocks = ['. ', '░ ', '▒ ', '█ ']

    print(f'  {"":9}', end='')
    for h in display_hours:
        print(f'{h:02d}   ', end='')
    print()

    for i, person in enumerate(participants):
        row = heatmap[i]
        row_max = row.max()
        print(f'  {person:<9}', end='')
        for h in display_hours:
            if row_max == 0:
                block = blocks[0]
            else:
                frac = row[h] / row_max
                if frac == 0 or frac < 0.25:
                    block = blocks[0]
                elif frac < 0.5:
                    block = blocks[1]
                elif frac < 0.75:
                    block = blocks[2]
                else:
                    block = blocks[3]
            print(f'{block}   ', end='')
        print()
    print()

    # ── SECTION 4: FAVOURITE WORDS ───────────────────────────────
    print(f'  ── THIS GROUP\'S FAVOURITE WORDS {"─" * (W - 34)}')
    top_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)[:10]
    max_wcount = top_words[0][1]
    for word, count in top_words:
        bar_len = int(count / max_wcount * 22)
        bar = '█' * bar_len
        print(f'  {word:<12} {bar:<24} {count}')
    print()

    # ── SECTION 5: RESPONSE PATTERNS ────────────────────────────
    print(f'  ── RESPONSE PATTERNS {"─" * (W - 23)}')
    sorted_resp = sorted(avg_response.items(), key=lambda x: x[1])
    fastest = sorted_resp[0]
    slowest = sorted_resp[-1]
    print(f'  Fastest replier : {fastest[0]} (avg {format_duration(fastest[1])})')
    print(f'  Slowest replier : {slowest[0]} (avg {format_duration(slowest[1])})')
    print()

    # ── SECTION 6: SILENT STREAKS ────────────────────────────────
    print(f'  ── LONGEST SILENT STREAKS {"─" * (W - 28)}')
    sorted_streaks = sorted(longest_streaks.items(), key=lambda x: x[1], reverse=True)
    for person, streak in sorted_streaks:
        start, end = streak_dates[person]
        if streak == 0:
            streak_str = '0 days (never went silent)'
        elif start and end:
            streak_str = f'{streak} days ({start.strftime("%d %b")} – {end.strftime("%d %b")})'
        else:
            streak_str = f'{streak} days'
        print(f'  {person:<8} : {streak_str}')
    print()

    # ── SECTION 7: PERSONALITY ARCHETYPES ───────────────────────
    print(f'  ── PERSONALITY ARCHETYPES {"─" * (W - 28)}')
    for person in participants:
        arch, raw_score = assigned_archetypes[person]
        desc_func = archetype_descriptions.get(arch, lambda p, s: f'score {s:.2f}')
        desc = desc_func(person, raw_score)
        print(f'  {person:<8} → {arch} ({desc})')
    print()

    # ── BONUS: MEDIA & DELETIONS ─────────────────────────────────
    if media_pp or deleted_pp:
        print(f'  ── BONUS STATS {"─" * (W - 17)}')
        if media_pp:
            print(f'  Media shared  : {", ".join(f"{p}={c}" for p, c in sorted(media_pp.items(), key=lambda x: -x[1]))}')
        if deleted_pp:
            print(f'  Msgs deleted  : {", ".join(f"{p}={c}" for p, c in sorted(deleted_pp.items(), key=lambda x: -x[1]))}')
        print()

    # ── FOOTER ──────────────────────────────────────────────────
    print('=' * W)
    print(f'{" Generated by GroupDNA • Built with Python + NumPy only ":^{W}}')
    print(f'{" No pandas. No matplotlib. No regex. Just fundamentals. ":^{W}}')
    print('=' * W)


# ── PRINT THE FINAL REPORT ───────────────────────────────────────
print_final_report()


# ============================================================
# ---
# ## Reflection
#
# **Hardest part:** Feature 6 (Response Times & Silent Streaks) — correctly walking through ordered timestamps and tracking per-person states without using any date library beyond `datetime.strptime` required careful loop design.
#
# **What I'd do differently:** Add emoji support in the tokenizer for richer word-frequency output when running on real chats. Also, the stop-words list is English-biased — a real deployment on a Hinglish group would need a combined Hindi+English stop-word list.
#
# **Key learning:** Real-world data is messy. The WhatsApp export has five different edge cases baked in — and handling them with only `str.split()`, `str.startswith()`, and conditionals (no regex!) is genuinely what production data parsing looks like at its core.
#
# **Constraint discipline:** Forbidden: `pandas`, `matplotlib`, `collections.Counter`, `re`. Used only: `numpy`, `datetime`. Everything else is vanilla Python. This is the part that impresses recruiters.
#
# ---
# *Built as part of The Unlox Academy Week 1 Minor Project.*

Imports ready. NumPy version: 2.0.2
Successfully parsed 3127 messages from 6 participants over 60 days
Skipped: 4 system messages, 32 media-omitted, 15 deleted messages
Participants: Aman, Karan, Neha, Priya, Rahul, Vikas

First 3 messages:
  [01/04/24, 01:17] Rahul: scene fix
  [01/04/24, 01:17] Rahul: haan
  [01/04/24, 01:18] Rahul: kya scene

Last 3 messages:
  [30/05/24, 21:17] Aman: the existential dread is back
  [30/05/24, 21:30] Karan: Long day guys, woke up at six for that placement workshop wh
  [30/05/24, 23:31] Aman: anyone awake?
                       GROUP OVERVIEW                       
  Group               : Hostel Bois 4ever
  Period              : 01 April 2024 to 30 May 2024 (60 days)
  Total messages      : 3,127
  Participants        : 6

  MESSAGES PER PERSON
  --------------------------------------------------------
  Rahul    ████████████████████    940 (30.1%)
  Priya    ███████████████.....    712 (22.8%)
  Neha     █████████████.......    624 (20.0%)
  Aman